# Components lab 04: detectors and receivers

Detector components turn arriving qstate-backed signals into reports. A report is not just “clicked”: it carries the qstate result, raw clicks, timing, flags, and the detector's decision.

In [ ]:
from __future__ import annotations

from dataclasses import dataclass, field

from simyuj.components import Port, PortDelivery, PortDirection, PortKind, connect_ports
from simyuj.components.detectors.detector_array import DetectorArray
from simyuj.components.detectors.primitives.actions import ACTION_DETECT_SIGNAL
from simyuj.components.detectors.primitives.params import SinglePhotonDetectorParams
from simyuj.components.detectors.primitives.reports import DetectionReport
from simyuj.components.detectors.primitives.rng import DetectorRNGStreams
from simyuj.components.detectors.single_photon import SinglePhotonDetector
from simyuj.engine import Component, Timeline
from simyuj.primitives.subsystems import SubsystemHandle
from simyuj.qstate import SubsystemId
from simyuj.runtime.binding import BindingContext
from simyuj.signal import EncodingScheme, Signal, SignalKind

## 1. A scripted RNG for one detector channel

This lets one cell show efficiency, dark counts, jitter, and dead time without hiding behind a full component.

In [ ]:
@dataclass(slots=True)
class ScriptedRNG:
    random_values: list[float] = field(default_factory=list)
    normal_values: list[float] = field(default_factory=list)
    poisson_values: list[int] = field(default_factory=list)

    def random(self) -> float:
        return self.random_values.pop(0) if self.random_values else 0.0

    def normal(self, *, loc: float, scale: float) -> float:
        return self.normal_values.pop(0) if self.normal_values else 0.0

    def poisson(self, lam: float) -> int:
        return self.poisson_values.pop(0) if self.poisson_values else 0


def streams(**overrides) -> DetectorRNGStreams:
    return DetectorRNGStreams(
        efficiency=overrides.get('efficiency', ScriptedRNG()),
        dark=overrides.get('dark', ScriptedRNG()),
        jitter=overrides.get('jitter', ScriptedRNG()),
        afterpulse=overrides.get('afterpulse', ScriptedRNG()),
    )

In [ ]:
detector = SinglePhotonDetector(
    detector_id='d0',
    params=SinglePhotonDetectorParams(
        efficiency=0.70,
        dark_count_rate_hz=1e12,
        dead_time_ticks=4,
        jitter_stddev_ticks=1.0,
    ),
)

rngs = streams(
    efficiency=ScriptedRNG(random_values=[0.2]),
    dark=ScriptedRNG(poisson_values=[1]),
    jitter=ScriptedRNG(normal_values=[1.0, 0.0]),
)


In [ ]:
clicks = detector.evaluate_window(
    time=100,
    signal_present=True,
    window_duration_ticks=5,
    rngs=rngs,
    outcome_label='0',
)

print('first window clicks:', len(clicks))
for click in clicks:
    print(' ', click)
print('dead until:', detector.dead_until)

blocked = detector.evaluate_window(
    time=102,
    signal_present=True,
    window_duration_ticks=5,
    rngs=streams(),
    outcome_label='0',
)
print('second window during dead time:', blocked)


## 2. Build a detector-array receiver

The readout map says which detector channel corresponds to each qstate result label.

In [ ]:
READOUT_ZX = {
    'z': {'0': 'd0', '1': 'd1'},
    'x': {'+': 'd0', '-': 'd1'},
}

@dataclass(slots=True)
class QuantumSource(Component):
    component_id: str
    output_port: Port = field(init=False)

    def __post_init__(self) -> None:
        self.output_port = Port('out', self, self.component_id, PortKind.QUANTUM, PortDirection.EGRESS)

    def handle_event(self, event, timeline) -> None:
        raise ValueError(event.action)


@dataclass(slots=True)
class ReportSink(Component):
    component_id: str
    input_port: Port = field(init=False)
    received: list[tuple[int, DetectionReport]] = field(default_factory=list)

    def __post_init__(self) -> None:
        self.input_port = Port('in', self, self.component_id, PortKind.CLASSICAL, PortDirection.INGRESS)

    def handle_event(self, event, timeline) -> None:
        delivery = event.payload_ref
        if not isinstance(delivery, PortDelivery):
            raise TypeError('expected PortDelivery')
        self.received.append((timeline.current_time, delivery.payload))

In [ ]:
def make_signal(timeline: Timeline, *, signal_id: str, state: str, time: int) -> Signal:
    subsystem = SubsystemId(f'{signal_id}:target')
    state_ref = timeline.qstate.prepare(state, subsystems=(subsystem,))
    return Signal(
        id=signal_id,
        signal_kind=SignalKind.PHOTON,
        encoding_scheme=EncodingScheme.POLARIZATION,
        emission_time=time,
        origin='source',
        state_ref=state_ref,
        state_targets=(SubsystemHandle(label=str(subsystem), kind='qubit', index=0),),
    )

In [ ]:
timeline = Timeline(master_seed=77)
source = QuantumSource('source')
report_sink = ReportSink('report.sink')

array = DetectorArray(
    device_id='bob.detector',
    detectors=(
        SinglePhotonDetector('d0', SinglePhotonDetectorParams(efficiency=0.85, dark_count_rate_hz=2e10, dead_time_ticks=3, jitter_stddev_ticks=1.0)),
        SinglePhotonDetector('d1', SinglePhotonDetectorParams(efficiency=0.85, dark_count_rate_hz=2e10, dead_time_ticks=3, jitter_stddev_ticks=1.0)),
    ),
    measurement='z',
    readout=READOUT_ZX,
    detection_window_ticks=6,
    output_latency_ticks=2,
)
array.bind(BindingContext(timeline=timeline, logger=timeline.logger))

connect_ports(source.output_port, array.input_port, target_action=ACTION_DETECT_SIGNAL)
connect_ports(array.output_port, report_sink.input_port, target_action='receive_detection_report')

print('detectors:', [det.detector_id for det in array.detectors])
print('consume signal:', array.consume_signal)
print('output latency ticks:', array.output_latency_ticks)

In [ ]:
for tick, state in [(10, '|0>'), (15, '|1>'), (18, '|+>')]:
    signal = make_signal(timeline, signal_id=f'sig-{tick}', state=state, time=tick)
    source.output_port.connection.transmit(signal, timeline, time=tick)

timeline.run_until_empty()

print('stored detector reports:', len(array.reports))
print('emitted reports:', len(report_sink.received))
print('live qstate records after consume_signal:', timeline.qstate.size())

In [ ]:
print('reports delivered:', len(report_sink.received))
for time, report in report_sink.received:
    click_summary = [
        (click.detector_id, click.time, click.trigger, click.outcome_label)
        for click in report.raw_clicks
    ]
    print(
        ' t=', time,
        'id=', report.report_id,
        'signal=', report.signal_id,
        'success=', report.success,
        'outcome=', report.outcome,
        'clicks=', click_summary,
    )


## 3. Keep qstate when the detector is only observing

`consume_signal=False` is useful when a receiver should report a measurement without deleting the target from qstate.

In [ ]:
timeline = Timeline(master_seed=77)
source = QuantumSource('source.keep')
report_sink = ReportSink('report.keep')
array = DetectorArray(
    device_id='observer.detector',
    detectors=(SinglePhotonDetector('d0'), SinglePhotonDetector('d1')),
    measurement='z',
    readout=READOUT_ZX,
    consume_signal=False,
)
array.bind(BindingContext(timeline=timeline, logger=timeline.logger))
connect_ports(source.output_port, array.input_port, target_action=ACTION_DETECT_SIGNAL)
connect_ports(array.output_port, report_sink.input_port, target_action='receive_detection_report')

signal = make_signal(timeline, signal_id='observed', state='|1>', time=0)
print('qstate size before:', timeline.qstate.size())
source.output_port.connection.transmit(signal, timeline, time=0)
timeline.run_until_empty()
print('qstate size after:', timeline.qstate.size())
print('report outcome:', report_sink.received[0][1].outcome)

## Keep this model in your head

Detector physics produces raw clicks. The detector component resolves those clicks into reports. Whether qstate is consumed is a component setting, not a property of the signal itself.